In [2]:
import pandas as pd
import numpy as np

file_path = r"raw_data\state_M2024_dl.xlsx"

oews_2024 = pd.read_excel(
    file_path,
    engine="openpyxl"
)

print("Shape:", oews_2024.shape)

print("\nColumns:")
print(oews_2024.columns.tolist())

oews_2024.head()

Shape: (37609, 32)

Columns:
['AREA', 'AREA_TITLE', 'AREA_TYPE', 'PRIM_STATE', 'NAICS', 'NAICS_TITLE', 'I_GROUP', 'OWN_CODE', 'OCC_CODE', 'OCC_TITLE', 'O_GROUP', 'TOT_EMP', 'EMP_PRSE', 'JOBS_1000', 'LOC_QUOTIENT', 'PCT_TOTAL', 'PCT_RPT', 'H_MEAN', 'A_MEAN', 'MEAN_PRSE', 'H_PCT10', 'H_PCT25', 'H_MEDIAN', 'H_PCT75', 'H_PCT90', 'A_PCT10', 'A_PCT25', 'A_MEDIAN', 'A_PCT75', 'A_PCT90', 'ANNUAL', 'HOURLY']


,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,21.07,30.82,47.51,23520,30660,43830,64110,98810,NaN,NaN
1,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,48.39,68.5,98.03,51100,72870,100640,142480,203900,NaN,NaN
2,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,79.04,106.69,#,104950,130950,164400,221910,#,NaN,NaN
3,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,51.12,78.26,#,50410,74720,106330,162780,#,NaN,NaN
4,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1031,Legislators,...,*,*,*,18270,20950,26990,41760,63900,True,NaN


In [3]:
# ==========================================
# OEWS 2024 STRUCTURE CHECK
# ==========================================

print("=== AREA TYPES ===")
print(
    oews_2024["AREA_TYPE"]
    .value_counts()
    .sort_index()
)


print("\n=== OCCUPATION GROUPS ===")
print(
    oews_2024["O_GROUP"]
    .value_counts(dropna=False)
)


print("\n=== INDUSTRY GROUPS ===")
print(
    oews_2024["I_GROUP"]
    .value_counts(dropna=False)
)


print("\n=== NAICS VALUES ===")
print(
    oews_2024["NAICS"]
    .value_counts()
    .head(20)
)


print("\n=== OWNERSHIP CODES ===")
print(
    oews_2024["OWN_CODE"]
    .value_counts()
    .sort_index()
)


print("\n=== NUMBER OF AREAS ===")
print(oews_2024["AREA"].nunique())


print("\n=== EXAMPLE OCCUPATION LEVELS ===")

print(
    oews_2024[
        ["OCC_CODE", "OCC_TITLE", "O_GROUP"]
    ]
    .drop_duplicates()
    .head(30)
    .to_string(index=False)
)

=== AREA TYPES ===
AREA_TYPE
2    36600
3     1009
Name: count, dtype: int64

=== OCCUPATION GROUPS ===
O_GROUP
detailed    36367
major        1188
total          54
Name: count, dtype: int64

=== INDUSTRY GROUPS ===
I_GROUP
cross-industry    37609
Name: count, dtype: int64

=== NAICS VALUES ===
NAICS
0    37609
Name: count, dtype: int64

=== OWNERSHIP CODES ===
OWN_CODE
1235    37609
Name: count, dtype: int64

=== NUMBER OF AREAS ===
54

=== EXAMPLE OCCUPATION LEVELS ===
OCC_CODE                                                     OCC_TITLE  O_GROUP
 00-0000                                               All Occupations    total
 11-0000                                        Management Occupations    major
 11-1011                                              Chief Executives detailed
 11-1021                               General and Operations Managers detailed
 11-1031                                                   Legislators detailed
 11-2011                           Advertis

In [4]:
print("=== AREAS ===")

areas = (
    oews_2024[
        ["AREA", "AREA_TITLE", "AREA_TYPE", "PRIM_STATE"]
    ]
    .drop_duplicates()
    .sort_values("AREA")
)

print(areas.to_string(index=False))


print("\n=== AREA TYPE COUNTS ===")

print(
    areas["AREA_TYPE"].value_counts().sort_index()
)


print("\n=== MAJOR OCCUPATION COUNT PER AREA ===")

major_counts = (
    oews_2024[
        oews_2024["O_GROUP"] == "major"
    ]
    .groupby(["AREA", "AREA_TITLE"])
    ["OCC_CODE"]
    .nunique()
)

print(major_counts)

=== AREAS ===
 AREA           AREA_TITLE  AREA_TYPE PRIM_STATE
    1              Alabama          2         AL
    2               Alaska          2         AK
    4              Arizona          2         AZ
    5             Arkansas          2         AR
    6           California          2         CA
    8             Colorado          2         CO
    9          Connecticut          2         CT
   10             Delaware          2         DE
   11 District of Columbia          2         DC
   12              Florida          2         FL
   13              Georgia          2         GA
   15               Hawaii          2         HI
   16                Idaho          2         ID
   17             Illinois          2         IL
   18              Indiana          2         IN
   19                 Iowa          2         IA
   20               Kansas          2         KS
   21             Kentucky          2         KY
   22            Louisiana          2         LA
   23 

In [5]:
# ==========================================
# BUILD OEWS STATE × OCCUPATION 2024
# ==========================================

# Keep only 50 states + DC
oews_state_occupation_2024 = oews_2024[
    oews_2024["AREA_TYPE"] == 2
].copy()

# Add year
oews_state_occupation_2024["YEAR"] = 2024


# ------------------------------------------
# SELECT FIELDS
# ------------------------------------------

columns_to_keep = [
    "YEAR",
    "AREA",
    "AREA_TITLE",
    "PRIM_STATE",
    "OCC_CODE",
    "OCC_TITLE",
    "O_GROUP",
    "TOT_EMP",
    "EMP_PRSE",
    "JOBS_1000",
    "LOC_QUOTIENT",
    "H_MEAN",
    "A_MEAN",
    "MEAN_PRSE",
    "H_PCT10",
    "H_PCT25",
    "H_MEDIAN",
    "H_PCT75",
    "H_PCT90",
    "A_PCT10",
    "A_PCT25",
    "A_MEDIAN",
    "A_PCT75",
    "A_PCT90"
]

oews_state_occupation_2024 = (
    oews_state_occupation_2024[columns_to_keep]
    .copy()
)


# ------------------------------------------
# CONVERT NUMERIC FIELDS
# ------------------------------------------

numeric_columns = [
    "TOT_EMP",
    "EMP_PRSE",
    "JOBS_1000",
    "LOC_QUOTIENT",
    "H_MEAN",
    "A_MEAN",
    "MEAN_PRSE",
    "H_PCT10",
    "H_PCT25",
    "H_MEDIAN",
    "H_PCT75",
    "H_PCT90",
    "A_PCT10",
    "A_PCT25",
    "A_MEDIAN",
    "A_PCT75",
    "A_PCT90"
]

for col in numeric_columns:
    oews_state_occupation_2024[col] = pd.to_numeric(
        oews_state_occupation_2024[col],
        errors="coerce"
    )


# ------------------------------------------
# RENAME FOR READABILITY
# ------------------------------------------

oews_state_occupation_2024 = (
    oews_state_occupation_2024.rename(
        columns={
            "YEAR": "year",
            "AREA": "area_code",
            "AREA_TITLE": "state",
            "PRIM_STATE": "state_abbr",
            "OCC_CODE": "occupation_code",
            "OCC_TITLE": "occupation",
            "O_GROUP": "occupation_level",
            "TOT_EMP": "employment",
            "EMP_PRSE": "employment_prse",
            "JOBS_1000": "jobs_per_1000",
            "LOC_QUOTIENT": "location_quotient",
            "H_MEAN": "mean_hourly_wage",
            "A_MEAN": "mean_annual_wage",
            "MEAN_PRSE": "mean_wage_prse",
            "H_PCT10": "hourly_p10",
            "H_PCT25": "hourly_p25",
            "H_MEDIAN": "median_hourly_wage",
            "H_PCT75": "hourly_p75",
            "H_PCT90": "hourly_p90",
            "A_PCT10": "annual_p10",
            "A_PCT25": "annual_p25",
            "A_MEDIAN": "median_annual_wage",
            "A_PCT75": "annual_p75",
            "A_PCT90": "annual_p90"
        }
    )
)


# ==========================================
# CHECK RESULT
# ==========================================

print("Shape:", oews_state_occupation_2024.shape)

print("\nOccupation levels:")
print(
    oews_state_occupation_2024[
        "occupation_level"
    ].value_counts()
)

print("\nStates:")
print(
    oews_state_occupation_2024[
        "state"
    ].nunique()
)

print("\nMissing values:")
print(
    oews_state_occupation_2024
    .isna()
    .sum()
)

oews_state_occupation_2024.head()

Shape: (36600, 24)

Occupation levels:
occupation_level
detailed    35427
major        1122
total          51
Name: count, dtype: int64

States:
51

Missing values:
year                     0
area_code                0
state                    0
state_abbr               0
occupation_code          0
occupation               0
occupation_level         0
employment            1319
employment_prse       1319
jobs_per_1000         1319
location_quotient     1319
mean_hourly_wage      3060
mean_annual_wage       617
mean_wage_prse         398
hourly_p10            3060
hourly_p25            3209
median_hourly_wage    3493
hourly_p75            3761
hourly_p90            4237
annual_p10             617
annual_p25             766
median_annual_wage    1060
annual_p75            1346
annual_p90            1909
dtype: int64


,year,area_code,state,state_abbr,occupation_code,occupation,occupation_level,employment,employment_prse,jobs_per_1000,...,hourly_p10,hourly_p25,median_hourly_wage,hourly_p75,hourly_p90,annual_p10,annual_p25,median_annual_wage,annual_p75,annual_p90
0,2024,1,Alabama,AL,00-0000,All Occupations,total,2091480.0,0.0,1000.000,...,11.31,14.74,21.07,30.82,47.51,23520.0,30660.0,43830.0,64110.0,98810.0
1,2024,1,Alabama,AL,11-0000,Management Occupations,major,110240.0,0.9,52.708,...,24.57,35.03,48.39,68.50,98.03,51100.0,72870.0,100640.0,142480.0,203900.0
2,2024,1,Alabama,AL,11-1011,Chief Executives,detailed,830.0,17.6,0.396,...,50.46,62.96,79.04,106.69,NaN,104950.0,130950.0,164400.0,221910.0,NaN
3,2024,1,Alabama,AL,11-1021,General and Operations Managers,detailed,32370.0,1.6,15.476,...,24.24,35.92,51.12,78.26,NaN,50410.0,74720.0,106330.0,162780.0,NaN
4,2024,1,Alabama,AL,11-1031,Legislators,detailed,1120.0,8.7,0.533,...,NaN,NaN,NaN,NaN,NaN,18270.0,20950.0,26990.0,41760.0,63900.0


In [6]:
# ==========================================
# OEWS 2024 QUALITY CONTROL
# ==========================================

print("=== 1. DUPLICATES ===")

duplicates = (
    oews_state_occupation_2024
    .duplicated(
        subset=[
            "year",
            "state",
            "occupation_code"
        ]
    )
    .sum()
)

print("Duplicates:", duplicates)


print("\n=== 2. MISSING VALUES BY OCCUPATION LEVEL ===")

important_metrics = [
    "employment",
    "jobs_per_1000",
    "location_quotient",
    "mean_annual_wage",
    "median_annual_wage"
]

missing_by_level = (
    oews_state_occupation_2024
    .groupby("occupation_level")[important_metrics]
    .agg(lambda x: x.isna().sum())
)

print(missing_by_level)


print("\n=== 3. MAJOR GROUP COMPLETENESS ===")

major = oews_state_occupation_2024[
    oews_state_occupation_2024["occupation_level"] == "major"
]

print("Major rows:", len(major))

print("\nMissing major employment:")
print(major["employment"].isna().sum())

print("Missing major median annual wage:")
print(major["median_annual_wage"].isna().sum())

print("Missing major location quotient:")
print(major["location_quotient"].isna().sum())


print("\n=== 4. EMPLOYMENT RANGE ===")

print(
    oews_state_occupation_2024[
        "employment"
    ].describe()
)


print("\n=== 5. ANNUAL WAGE RANGE ===")

print(
    oews_state_occupation_2024[
        "median_annual_wage"
    ].describe()
)


print("\n=== 6. STATES WITH MOST MISSING DETAILED EMPLOYMENT ===")

detailed = oews_state_occupation_2024[
    oews_state_occupation_2024["occupation_level"] == "detailed"
]

missing_detailed = (
    detailed
    .groupby("state")["employment"]
    .apply(lambda x: x.isna().sum())
    .sort_values(ascending=False)
)

print(missing_detailed.head(15))

=== 1. DUPLICATES ===
Duplicates: 0

=== 2. MISSING VALUES BY OCCUPATION LEVEL ===
                  employment  jobs_per_1000  location_quotient  \
occupation_level                                                 
detailed                1319           1319               1319   
major                      0              0                  0   
total                      0              0                  0   

                  mean_annual_wage  median_annual_wage  
occupation_level                                        
detailed                       616                1059  
major                            1                   1  
total                            0                   0  

=== 3. MAJOR GROUP COMPLETENESS ===
Major rows: 1122

Missing major employment:
0
Missing major median annual wage:
1
Missing major location quotient:
0

=== 4. EMPLOYMENT RANGE ===
count    3.528100e+04
mean     1.307908e+04
std      1.788797e+05
min      3.000000e+01
25%      2.400000e+02
50%     

In [7]:
import pandas as pd
import numpy as np
import os

# ==========================================
# OEWS 2021–2025 PIPELINE
# ==========================================

data_path = r"raw_data"

years = range(2021, 2026)

all_oews = []

numeric_columns = [
    "TOT_EMP",
    "EMP_PRSE",
    "JOBS_1000",
    "LOC_QUOTIENT",
    "H_MEAN",
    "A_MEAN",
    "MEAN_PRSE",
    "H_PCT10",
    "H_PCT25",
    "H_MEDIAN",
    "H_PCT75",
    "H_PCT90",
    "A_PCT10",
    "A_PCT25",
    "A_MEDIAN",
    "A_PCT75",
    "A_PCT90"
]

columns_to_keep = [
    "AREA",
    "AREA_TITLE",
    "PRIM_STATE",
    "OCC_CODE",
    "OCC_TITLE",
    "O_GROUP",
    "TOT_EMP",
    "EMP_PRSE",
    "JOBS_1000",
    "LOC_QUOTIENT",
    "H_MEAN",
    "A_MEAN",
    "MEAN_PRSE",
    "H_PCT10",
    "H_PCT25",
    "H_MEDIAN",
    "H_PCT75",
    "H_PCT90",
    "A_PCT10",
    "A_PCT25",
    "A_MEDIAN",
    "A_PCT75",
    "A_PCT90"
]


for year in years:

    print(f"Processing {year}...")

    file_path = os.path.join(
        data_path,
        f"state_M{year}_dl.xlsx"
    )

    df = pd.read_excel(
        file_path,
        engine="openpyxl"
    )

    # --------------------------------------
    # Keep 50 states + DC only
    # --------------------------------------

    df = df[
        df["AREA_TYPE"] == 2
    ].copy()

    # --------------------------------------
    # Select required fields
    # --------------------------------------

    df = df[columns_to_keep].copy()

    # Add year
    df["year"] = year

    # --------------------------------------
    # Convert BLS numeric fields
    # *, #, etc. → NaN
    # --------------------------------------

    for col in numeric_columns:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    # --------------------------------------
    # Rename columns
    # --------------------------------------

    df = df.rename(
        columns={
            "AREA": "area_code",
            "AREA_TITLE": "state",
            "PRIM_STATE": "state_abbr",
            "OCC_CODE": "occupation_code",
            "OCC_TITLE": "occupation",
            "O_GROUP": "occupation_level",
            "TOT_EMP": "employment",
            "EMP_PRSE": "employment_prse",
            "JOBS_1000": "jobs_per_1000",
            "LOC_QUOTIENT": "location_quotient",
            "H_MEAN": "mean_hourly_wage",
            "A_MEAN": "mean_annual_wage",
            "MEAN_PRSE": "mean_wage_prse",
            "H_PCT10": "hourly_p10",
            "H_PCT25": "hourly_p25",
            "H_MEDIAN": "median_hourly_wage",
            "H_PCT75": "hourly_p75",
            "H_PCT90": "hourly_p90",
            "A_PCT10": "annual_p10",
            "A_PCT25": "annual_p25",
            "A_MEDIAN": "median_annual_wage",
            "A_PCT75": "annual_p75",
            "A_PCT90": "annual_p90"
        }
    )

    # --------------------------------------
    # Column order
    # --------------------------------------

    df = df[
        [
            "year",
            "area_code",
            "state",
            "state_abbr",
            "occupation_code",
            "occupation",
            "occupation_level",
            "employment",
            "employment_prse",
            "jobs_per_1000",
            "location_quotient",
            "mean_hourly_wage",
            "mean_annual_wage",
            "mean_wage_prse",
            "hourly_p10",
            "hourly_p25",
            "median_hourly_wage",
            "hourly_p75",
            "hourly_p90",
            "annual_p10",
            "annual_p25",
            "median_annual_wage",
            "annual_p75",
            "annual_p90"
        ]
    ]

    all_oews.append(df)

    print(
        f"  Rows: {len(df):,} | "
        f"States: {df['state'].nunique()} | "
        f"Major: {(df['occupation_level'] == 'major').sum():,} | "
        f"Detailed: {(df['occupation_level'] == 'detailed').sum():,}"
    )


# ==========================================
# COMBINE ALL YEARS
# ==========================================

oews_state_occupation = pd.concat(
    all_oews,
    ignore_index=True
)


print("\nDONE")

print(
    "Final shape:",
    oews_state_occupation.shape
)

print("\nRows by year:")
print(
    oews_state_occupation[
        "year"
    ].value_counts().sort_index()
)

print("\nOccupation levels by year:")
print(
    pd.crosstab(
        oews_state_occupation["year"],
        oews_state_occupation["occupation_level"]
    )
)

Processing 2021...
  Rows: 36,561 | States: 51 | Major: 1,122 | Detailed: 35,388
Processing 2022...
  Rows: 36,574 | States: 51 | Major: 1,122 | Detailed: 35,401
Processing 2023...
  Rows: 36,643 | States: 51 | Major: 1,122 | Detailed: 35,470
Processing 2024...
  Rows: 36,600 | States: 51 | Major: 1,122 | Detailed: 35,427
Processing 2025...
  Rows: 36,396 | States: 51 | Major: 1,122 | Detailed: 35,223

DONE
Final shape: (182774, 24)

Rows by year:
year
2021    36561
2022    36574
2023    36643
2024    36600
2025    36396
Name: count, dtype: int64

Occupation levels by year:
occupation_level  detailed  major  total
year                                    
2021                 35388   1122     51
2022                 35401   1122     51
2023                 35470   1122     51
2024                 35427   1122     51
2025                 35223   1122     51


In [8]:
# ==========================================
# OEWS 2021–2025 FINAL QUALITY CONTROL
# ==========================================

print("=== 1. DUPLICATES ===")

duplicates = (
    oews_state_occupation
    .duplicated(
        subset=[
            "year",
            "state",
            "occupation_code"
        ]
    )
    .sum()
)

print("Duplicates:", duplicates)


# ------------------------------------------

print("\n=== 2. STATES PER YEAR ===")

print(
    oews_state_occupation
    .groupby("year")["state"]
    .nunique()
)


# ------------------------------------------

print("\n=== 3. MAJOR OCCUPATION CODES BY YEAR ===")

major_codes = {}

for year in range(2021, 2026):

    codes = sorted(
        oews_state_occupation.loc[
            (oews_state_occupation["year"] == year) &
            (oews_state_occupation["occupation_level"] == "major"),
            "occupation_code"
        ].unique()
    )

    major_codes[year] = set(codes)

    print(year, codes)


# ------------------------------------------

print("\n=== 4. ARE MAJOR CODES IDENTICAL? ===")

base = major_codes[2021]

for year in range(2022, 2026):

    print(
        f"2021 vs {year}:",
        base == major_codes[year]
    )


# ------------------------------------------

print("\n=== 5. MISSING MAJOR METRICS BY YEAR ===")

major = oews_state_occupation[
    oews_state_occupation["occupation_level"] == "major"
]

major_missing = (
    major
    .groupby("year")
    [
        [
            "employment",
            "jobs_per_1000",
            "location_quotient",
            "mean_annual_wage",
            "median_annual_wage"
        ]
    ]
    .agg(lambda x: x.isna().sum())
)

print(major_missing)


# ------------------------------------------

print("\n=== 6. MISSING DETAILED EMPLOYMENT BY YEAR ===")

detailed = oews_state_occupation[
    oews_state_occupation["occupation_level"] == "detailed"
]

print(
    detailed
    .groupby("year")["employment"]
    .agg(
        rows="size",
        missing=lambda x: x.isna().sum()
    )
)


# ------------------------------------------

print("\n=== 7. UNIQUE DETAILED OCCUPATION CODES ===")

for year in range(2021, 2026):

    count = (
        detailed.loc[
            detailed["year"] == year,
            "occupation_code"
        ]
        .nunique()
    )

    print(year, count)


# ------------------------------------------

print("\n=== 8. OCCUPATION CODE/TITLE CONSISTENCY ===")

occupation_titles = (
    oews_state_occupation[
        [
            "occupation_code",
            "occupation"
        ]
    ]
    .drop_duplicates()
    .groupby("occupation_code")["occupation"]
    .nunique()
)

print(
    "Codes with more than one title:",
    (occupation_titles > 1).sum()
)

print(
    occupation_titles[
        occupation_titles > 1
    ]
)

=== 1. DUPLICATES ===
Duplicates: 0

=== 2. STATES PER YEAR ===
year
2021    51
2022    51
2023    51
2024    51
2025    51
Name: state, dtype: int64

=== 3. MAJOR OCCUPATION CODES BY YEAR ===
2021 ['11-0000', '13-0000', '15-0000', '17-0000', '19-0000', '21-0000', '23-0000', '25-0000', '27-0000', '29-0000', '31-0000', '33-0000', '35-0000', '37-0000', '39-0000', '41-0000', '43-0000', '45-0000', '47-0000', '49-0000', '51-0000', '53-0000']
2022 ['11-0000', '13-0000', '15-0000', '17-0000', '19-0000', '21-0000', '23-0000', '25-0000', '27-0000', '29-0000', '31-0000', '33-0000', '35-0000', '37-0000', '39-0000', '41-0000', '43-0000', '45-0000', '47-0000', '49-0000', '51-0000', '53-0000']
2023 ['11-0000', '13-0000', '15-0000', '17-0000', '19-0000', '21-0000', '23-0000', '25-0000', '27-0000', '29-0000', '31-0000', '33-0000', '35-0000', '37-0000', '39-0000', '41-0000', '43-0000', '45-0000', '47-0000', '49-0000', '51-0000', '53-0000']
2024 ['11-0000', '13-0000', '15-0000', '17-0000', '19-0000', '2

In [9]:
output_file = (
    r"raw_data"
    r"\oews_state_occupation_2021_2025.csv"
)

oews_state_occupation.to_csv(
    output_file,
    index=False
)

print("OEWS file saved successfully.")
print("Rows:", len(oews_state_occupation))
print("Columns:", len(oews_state_occupation.columns))

OEWS file saved successfully.
Rows: 182774
Columns: 24
